In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
from libpysal import graph
from math import sqrt
import warnings
warnings.filterwarnings('ignore')

In [3]:
sizes = [400,100,25]
shapes = ["square","pent","hex"]

In [4]:
def generate_square_lattice(l):    
    l = np.arange(l)
    xs, ys = np.meshgrid(l, l)
    polys = []
    for x, y in zip(xs.flatten(), ys.flatten()):
        poly = Polygon([(x, y), (x + 1, y), (x + 1, y + 1), (x, y + 1)])
        polys.append(poly)
    polys = gpd.GeoSeries(polys)
    
    gdf = gpd.GeoDataFrame(
        {
            "geometry": polys},
        index=range(len(polys))      )
    
    return gdf


def generate_hex_lattice(cols, rows, size=1):
    w = np.sqrt(3) * size
    h = 2 * size
    vert_step = 1.5 * size
    horiz_step = w  
    lines = []
    
    for row in range(rows):
        for col in range(cols):
            x_offset = (w / 2) if (row % 2 == 1) else 0
            cx = (col * horiz_step) + x_offset
            cy = row * vert_step

            angles = np.radians([30, 90, 150, 210, 270, 330])
            vx = cx + size * np.cos(angles)
            vy = cy + size * np.sin(angles)
            
            for i in range(6):
                start_pt = (vx[i], vy[i])
                end_pt = (vx[(i+1)%6], vy[(i+1)%6])
                lines.append(LineString([start_pt, end_pt]))

    mesh = unary_union(lines)
    polys = list(polygonize(mesh))

    gdf = gpd.GeoDataFrame(
        {"geometry": polys},
        index=range(len(polys))
    )
    
    return gdf


def generate_pent_lattice(width, height, angle_deg=15, scale=1.0):  
    seeds = []
    theta = np.radians(angle_deg)
    c, s = np.cos(theta), np.sin(theta)
    r = 0.3 * scale 
    pad = 1
    
    for x in range(-pad, width + pad):
        for y in range(-pad, height + pad):
            cx, cy = x * scale, y * scale
            
            offsets = [
                (r, r), 
                (-r, r), 
                (-r, -r), 
                (r, -r)
            ]
            
            for ox, oy in offsets:
                # Rotate
                rox = ox * c - oy * s
                roy = ox * s + oy * c                
                seeds.append([cx + rox, cy + roy])

    seeds = np.array(seeds)
    vor = Voronoi(seeds)
    polys = []
    
    for region_index in vor.point_region:
        region = vor.regions[region_index]

        if -1 in region or len(region) == 0:
            continue
            
        # Get vertices for this region
        vertices = vor.vertices[region]
        polys.append(Polygon(vertices))

    clip_box = box(0, 0, (width-1) * scale, (height-1) * scale)
    gs = gpd.GeoSeries(polys)
    gs_clipped = gs.intersection(clip_box)
    gs_clipped = gs_clipped[~gs_clipped.is_empty & (gs_clipped.area > (0.1 * scale**2))]

    gdf = gpd.GeoDataFrame(
        {"geometry": gs_clipped}
        
    )
    gdf.reset_index(drop=True, inplace=True)
    
    return gdf

In [5]:
def extract_pent_subset(master_gdf, subset_cols, subset_rows, total_cols=20, total_rows=20):

    minx, miny, maxx, maxy = master_gdf.total_bounds
    
    x_step = (maxx - minx) / total_cols
    y_step = (maxy - miny) / total_rows
    
    cutoff_x = minx + (x_step * subset_cols) 
    cutoff_y = miny + (y_step * subset_rows) 
    
    mask = (master_gdf.centroid.x < cutoff_x) & (master_gdf.centroid.y < cutoff_y)
    
    subset_gdf = master_gdf[mask].copy()
    
    return subset_gdf
    

def reindex(gdf, row_size):

    bounds = gdf.bounds
    
    # 'miny' is the absolute lowest point; 'minx' is the absolute furthest left
    gdf['bottom_y'] = bounds['miny']
    gdf['left_x'] = bounds['minx']
    
        # 1. Fill the column with pandas' official "missing" value
    gdf['id'] = pd.NA

    gdf['id'] = gdf['id'].astype('Int64')
    
    gdf.sort_values(by='bottom_y', ascending=True, inplace=True)
    
    current_id = 0
    
    # Loop through the dataframe in chunks of 'row_size' (e.g., 20 at a time)
    for i in range(0, len(gdf), row_size):
        
        # A. Grab the next chunk of polygons (e.g., positions 0-19, then 20-39)
        chunk = gdf.iloc[i : i + row_size]
        
        # B. Sort just this chunk from left to right
        chunk_sorted = chunk.sort_values(by='left_x', ascending=True)
        
        # C. Generate the list of IDs for this chunk (e.g., 0 to 19)
        # Using len() ensures we don't break if the very top row has fewer than 20 polygons
        new_ids = range(current_id, current_id + len(chunk_sorted))
        
        # D. Assign these new IDs back to the main DataFrame safely using .loc
        # We match them using the index labels of the sorted chunk
        gdf.loc[chunk_sorted.index, 'id'] = new_ids
        
        # E. Update the counter for the next loop (so the next row starts at 20)
        current_id += len(chunk_sorted)

    # 5. Clean up: Sort the final dataset by the new 'id' and reset the index
    # 4. Clean up IN-PLACE
    gdf.sort_values(by='id', inplace=True)
    gdf.reset_index(drop=True, inplace=True)
    gdf.drop(columns=['bottom_y', 'left_x'], inplace=True)

In [6]:
def build_g_rook(gdf):
    return graph.Graph.build_contiguity(gdf, rook=True)

In [7]:
shape = "square"
for size in sizes:
    gdf = generate_square_lattice(sqrt(size))
    g_rook = build_g_rook(gdf)
    g_rook.to_parquet(f"graphs/{shape}/size_{size}/g_true.parquet")
    gdf.to_parquet(f"data/autocorrelation/gdf_{shape}_{size}.parquet")

    if size != 25:
        gdf.to_parquet(f"data/regionalization/gdf_{shape}_{size}.parquet")

In [8]:
shape = "pent"
pent_gdfs = {}
for size in sizes:
    if size == 400:
        pent_gdfs[size] = generate_pent_lattice(11,11) #trial and error, it is actually 20x20
    else:
        pent_gdfs[size] = extract_pent_subset(pent_gdfs[400],sqrt(size),sqrt(size))
        
    gdf = pent_gdfs[size]
    reindex(gdf,int(sqrt(size)))
    
    g_rook = build_g_rook(gdf)
    g_rook.to_parquet(f"graphs/{shape}/size_{size}/g_true.parquet")
    gdf.to_parquet(f"data/autocorrelation/gdf_{shape}_{size}.parquet")
    if size != 25:
        gdf.to_parquet(f"data/regionalization/gdf_{shape}_{size}.parquet")

In [9]:
shape = "hex"
for size in sizes:
    gdf = generate_hex_lattice(int(sqrt(size)),int(sqrt(size)))
    g_rook = build_g_rook(gdf)
    g_rook.to_parquet(f"graphs/{shape}/size_{size}/g_true.parquet")
    gdf.to_parquet(f"data/autocorrelation/gdf_{shape}_{size}.parquet")

    if size != 25:
        gdf.to_parquet(f"data/regionalization/gdf_{shape}_{size}.parquet")